In [1]:
%%capture
import warnings
warnings.filterwarnings('ignore')
import calitp_data_analysis.magics

import speedmap_utils
from shared_utils import webmap_utils, catalog_utils, rt_utils
import pandas as pd
import datetime as dt
import numpy as np

from functools import cache

from calitp_data_analysis.gcs_geopandas import GCSGeoPandas
from calitp_data_analysis.gcs_pandas import GCSPandas
import scipy
import geopandas as gpd

@cache
def gcs_pandas():
    return GCSPandas()

@cache
def gcs_geopandas():
    return GCSGeoPandas()

catalog = catalog_utils.get_catalog("gtfs_analytics_data")

In [2]:
SPEED_TRIPS_PATH = f"{catalog.speedmap_segments.dir}{catalog.speedmap_segments.stage4}"
SPEED_SEGS_PATH = f"{catalog.speedmap_segments.dir}{catalog.speedmap_segments.segment_timeofday}"

In [3]:
from update_vars_index import ANALYSIS_DATE_LIST
analysis_date = ANALYSIS_DATE_LIST[0]

In [4]:
analysis_date

'2026-08-12'

In [5]:
noncurrent_operators = pd.read_parquet('./_rt_progress_2026-08-12.parquet').query('analysis_date != @analysis_date')
noncurrent_operators

,analysis_name,name,base64_url,caltrans_district,analysis_date,status
27,City of Elk Grove,Elk Grove Schedule,aHR0cHM6Ly9pcG9ydGFsLnNhY3J0LmNvbS9ndGZzL2VnL2...,03 - Marysville / Sacramento,2026-07-15,speedmap_segs_available
44,Santa Barbara Metropolitan Transit District,SBMTD Schedule,aHR0cHM6Ly9zYm10ZC5nb3YvZ29vZ2xlX3RyYW5zaXQvZm...,05 - San Luis Obispo / Santa Barbara,2026-07-15,speedmap_segs_available
50,Long Beach Transit,Long Beach Schedule,aHR0cHM6Ly9kcml2ZS5nb29nbGUuY29tL3VjP2V4cG9ydD...,07 - Los Angeles / Ventura,2026-07-15,speedmap_segs_available
57,North County Transit District,North County Schedule,aHR0cHM6Ly9sZnBvcnRhbC5uY3RkLm9yZy9zdGF0aWNHVE...,11 - San Diego,2026-07-15,speedmap_segs_available
19,City of Fairfield,Bay Area 511 Fairfield and Suisun Transit Sche...,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,04 - Bay Area / Oakland,2026-06-10,speedmap_segs_available
41,Santa Cruz Metro,Bay Area 511 Santa Cruz Metro Schedule,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,05 - San Luis Obispo / Santa Barbara,2026-06-10,speedmap_segs_available
48,"San Diego Metropolitan Transit System, Airport...",San Diego Schedule,aHR0cHM6Ly93d3cuc2RtdHMuY29tL2dvb2dsZV90cmFuc2...,11 - San Diego,2026-06-10,speedmap_segs_available
97,None,SLO Peak Transit Schedule,aHR0cDovL2RhdGEucGVha3RyYW5zaXQuY29tL3N0YXRpY2...,05 - San Luis Obispo / Santa Barbara,2026-06-10,speedmap_segs_available


In [6]:
path = f"{SPEED_SEGS_PATH}_{'2026-06-10'}.parquet"
previous_segs = gcs_geopandas().read_parquet(path, filters=[["time_of_day", "==", "All Day"]])  # aggregated

### lookback, patch missing data

In [62]:
prev_seg_speeds = []
prev_trip_speeds = []
for previous_date in ['2026-06-10', '2026-07-15']:
    path = f"{SPEED_SEGS_PATH}_{previous_date}.parquet"
    previous_segs = gcs_geopandas().read_parquet(path, filters=[["time_of_day", "==", "All Day"]])  # aggregated
    previous_segs = previous_segs[previous_segs.base64_url.isin(
        noncurrent_operators[noncurrent_operators.analysis_date == previous_date].base64_url)]
    trip_speeds = gcs_pandas().read_parquet(f'{SPEED_TRIPS_PATH}_{previous_date}.parquet')
    trip_speeds = trip_speeds.merge(previous_segs[['segment_id', 'schedule_gtfs_dataset_key']], on=['segment_id', 'schedule_gtfs_dataset_key'])
    prev_seg_speeds += [previous_segs]
    prev_trip_speeds += [trip_speeds]

In [13]:
path = f"{SPEED_SEGS_PATH}_{analysis_date}.parquet"
speedmap_segs = gcs_geopandas().read_parquet(path, filters=[["time_of_day", "==", "All Day"]])  # aggregated

In [14]:
speedmap_segs = pd.concat([speedmap_segs] + prev_dfs)

In [16]:
speedmap_segs[speedmap_segs.analysis_name.isna()].name.unique() # TODO analysis_name guidance/add...

array(['Clovis PassioGo Schedule', 'Mountain Transit GMV Schedule',
       'Desert Roadrunner GMV Schedule', 'LAX Shuttles Schedule',
       'Victor Valley GMV Schedule',
       'University of San Diego Tram Services Schedule',
       'Basin Transit GMV Schedule', 'Merced GMV Schedule',
       'Tahoe Transportation District GMV Schedule',
       'Yuba-Sutter PassioGo Schedule',
       'Dana Point Trolley PassioGo Schedule',
       'SLO Peak Transit Schedule'], dtype=object)

In [21]:
msg = "no cols besides route_short_name, direction_id should be nan"
assert speedmap_segs.drop(columns=["route_short_name", "direction_id", "analysis_name", 
                                   "source_record_id"]).isna().any().any() == False, msg

In [22]:
shs = gcs_geopandas().read_parquet(rt_utils.SHN_PATH)

In [23]:
from calitp_data_analysis.geography_utils import CA_NAD83Albers_m

## Spatial Operations

In [24]:
speedmap_segs = speedmap_segs.to_crs(CA_NAD83Albers_m)
shs = shs.to_crs(CA_NAD83Albers_m)

### add linear intersect meters with shs

In [25]:
linear_intersect = speedmap_segs.copy().overlay(shs, how='intersection')
linear_intersect = linear_intersect.assign(approx_shs_intersect_meters = linear_intersect.geometry.map(lambda x: x.length))

In [26]:
# linear_intersect.sample(5000).explore()

In [27]:
linear_intersect = linear_intersect[['schedule_gtfs_dataset_key', 'segment_id', 'approx_shs_intersect_meters']].round(1)

In [28]:
linear_intersect.shape

(60762, 3)

In [29]:
# keep longest intersect per segment
linear_intersect = linear_intersect.groupby(['schedule_gtfs_dataset_key', 'segment_id']).max().reset_index()

In [30]:
linear_intersect.shape

(35158, 3)

In [31]:
# speedmap_segs = speedmap_segs.merge(linear_intersect, on = ['schedule_gtfs_dataset_key', 'segment_id'])

### Final sjoin

In [32]:
shs_segs = gpd.sjoin(speedmap_segs, shs, how='inner', predicate='intersects')

In [33]:
shs_segs.shape

(60762, 32)

In [34]:
shs_segs.merge(linear_intersect, on = ['schedule_gtfs_dataset_key', 'segment_id']).shape

(60762, 33)

In [35]:
shs_segs = shs_segs.merge(linear_intersect, on = ['schedule_gtfs_dataset_key', 'segment_id'])

In [ ]:
shs_segs = speedmap_utils.prepare_segment_gdf(shs_segs)
# shs_segs = speedmap_utils.prepare_segment_gdf(shs_segs).assign(analysis_date=analysis_date)

# Clean trip speeds, calculate delay based on p20 times

In [117]:
df = shs_segs.copy()

In [ ]:
prev_dfs = []
for previous_date in ['2026-06-10', '2026-07-15']:
    path = f"{SPEED_SEGS_PATH}_{previous_date}.parquet"
    previous_segs = gcs_geopandas().read_parquet(path, filters=[["time_of_day", "==", "All Day"]])  # aggregated
    previous_segs = previous_segs[previous_segs.base64_url.isin(
        noncurrent_operators[noncurrent_operators.analysis_date == previous_date].base64_url)]
    prev_dfs += [previous_segs]

In [63]:
trip_speeds = gcs_pandas().read_parquet(f'{SPEED_TRIPS_PATH}_{analysis_date}.parquet')

In [64]:
trip_speeds.shape

(2684459, 23)

In [66]:
trip_speeds = pd.concat([trip_speeds] + prev_trip_speeds)

In [67]:
# https://www.geeksforgeeks.org/machine-learning/z-score-for-outlier-detection-python/

trip_speeds = trip_speeds[['segment_id', 'sec_elapsed', 'schedule_gtfs_dataset_key', 'speed_mph']]
trip_speeds = trip_speeds.dropna()
# trip_speeds['speed_mph_z_score'] = scipy.stats.zscore(trip_speeds.speed_mph)
trip_speeds['sec_elapsed_z_score'] = scipy.stats.zscore(trip_speeds.sec_elapsed)

In [68]:
trip_speeds.head(5)

,segment_id,sec_elapsed,schedule_gtfs_dataset_key,speed_mph,sec_elapsed_z_score
0,6079-6082-1,44.0,076e30b080fdc5501151bd3fb0a37b9e,12.395530,-0.067136
1,6082-6085-1,34.0,076e30b080fdc5501151bd3fb0a37b9e,10.403270,-0.075686
2,6085-3401-1,103.0,076e30b080fdc5501151bd3fb0a37b9e,12.643995,-0.016691
3,3401-3404-1,66.0,076e30b080fdc5501151bd3fb0a37b9e,13.324467,-0.048326
4,3404-3407-1,53.0,076e30b080fdc5501151bd3fb0a37b9e,11.678617,-0.059441


In [69]:
# don't use z score for speeds

outliers = trip_speeds[(trip_speeds.speed_mph <= 0.1) | (trip_speeds.speed_mph > 80) | (trip_speeds.sec_elapsed_z_score.abs() >= 3)]
trip_speeds = trip_speeds[(trip_speeds.speed_mph >= 0.1) & (trip_speeds.speed_mph < 80) & (trip_speeds.sec_elapsed != 0)]
trip_speeds = trip_speeds[trip_speeds.sec_elapsed_z_score.abs() < 3]

In [70]:
trip_speeds.sort_values('speed_mph').head(3)

,segment_id,sec_elapsed,schedule_gtfs_dataset_key,speed_mph,sec_elapsed_z_score
727085,15566-15572-1,30.0,502fcdd340a7c5a4298bf36e2caacd19,0.100321,-0.079106
454463,144354-144449-1,80.0,3c4e82c605c871049b2f86993922a6ee,0.100446,-0.036356
260942,881872-881873-1,3270.0,2391c4d16e8250d3e31bac5057f99fbf,0.100652,2.691095


In [71]:
# outliers

In [90]:
trip_speeds.groupby(['segment_id', 'schedule_gtfs_dataset_key']).size()

segment_id           schedule_gtfs_dataset_key       
0-508-1              eacc5d61a5da62c7f29e4c6e080327b3      3
0002-0003-1          d27fcd9eec0803cce4dfdd2105aa5e8c     50
0002-0004-1          8e5034b0dddc8e5b0b0f04def813f271    348
0002-2288-1          076e30b080fdc5501151bd3fb0a37b9e      3
0002-6774-1          076e30b080fdc5501151bd3fb0a37b9e      4
                                                        ... 
warrbanc-intehous-1  448256d45dcb568a43092a5970cf77f4     11
warrchan-warrbanc-1  448256d45dcb568a43092a5970cf77f4     12
wescircl-westcres-1  448256d45dcb568a43092a5970cf77f4      8
westcres-shatalls-1  448256d45dcb568a43092a5970cf77f4     10
wursterh-hrstmemo-1  448256d45dcb568a43092a5970cf77f4     25
Length: 112657, dtype: int64

In [72]:
with_benchmark_times = (trip_speeds
    .groupby(['segment_id', 'schedule_gtfs_dataset_key'])
    .agg({"sec_elapsed": lambda x: np.quantile(x, q=.2), "speed_mph": lambda x: np.quantile(x, q=.8)})
    .rename(columns={'sec_elapsed': 'p20_seconds_benchmark', 'speed_mph': 'p80_mph_benchmark'})
    .reset_index()
    .merge(trip_speeds, on = ['segment_id', 'schedule_gtfs_dataset_key'])
)

In [73]:
with_benchmark_times.head(3)

,segment_id,schedule_gtfs_dataset_key,p20_seconds_benchmark,p80_mph_benchmark,sec_elapsed,speed_mph,sec_elapsed_z_score
0,0-508-1,eacc5d61a5da62c7f29e4c6e080327b3,30.0,38.177644,26.0,42.949849,-0.082526
1,0-508-1,eacc5d61a5da62c7f29e4c6e080327b3,30.0,38.177644,36.0,31.019335,-0.073976
2,0-508-1,eacc5d61a5da62c7f29e4c6e080327b3,30.0,38.177644,133.0,8.396211,0.008959


In [74]:
with_benchmark_times = with_benchmark_times.assign(transit_delay_sec = (with_benchmark_times.sec_elapsed - with_benchmark_times.p20_seconds_benchmark).clip(lower=0))

In [75]:
with_benchmark_times.head(3)

,segment_id,schedule_gtfs_dataset_key,p20_seconds_benchmark,p80_mph_benchmark,sec_elapsed,speed_mph,sec_elapsed_z_score,transit_delay_sec
0,0-508-1,eacc5d61a5da62c7f29e4c6e080327b3,30.0,38.177644,26.0,42.949849,-0.082526,0.0
1,0-508-1,eacc5d61a5da62c7f29e4c6e080327b3,30.0,38.177644,36.0,31.019335,-0.073976,6.0
2,0-508-1,eacc5d61a5da62c7f29e4c6e080327b3,30.0,38.177644,133.0,8.396211,0.008959,103.0


In [76]:
delay_by_segment = (with_benchmark_times
    .groupby(['segment_id', 'schedule_gtfs_dataset_key'])[['transit_delay_sec']]
    .sum()
    .rename(columns={'transit_delay_sec': 'total_transit_delay_sec'})
    .reset_index()
)
delay_by_segment = (delay_by_segment
                    .assign(transit_veh_hrs_delay = delay_by_segment.total_transit_delay_sec / 60**2)
                    .round(2)
)

In [77]:
delay_by_segment.head(3)

,segment_id,schedule_gtfs_dataset_key,total_transit_delay_sec,transit_veh_hrs_delay
0,0-508-1,eacc5d61a5da62c7f29e4c6e080327b3,109.0,0.03
1,0002-0003-1,d27fcd9eec0803cce4dfdd2105aa5e8c,2260.0,0.63
2,0002-0004-1,8e5034b0dddc8e5b0b0f04def813f271,14960.0,4.16


In [92]:
delay_by_segment.groupby(['segment_id', 'schedule_gtfs_dataset_key']).size().value_counts()

1    112657
Name: count, dtype: int64

In [118]:
df.shape

(60762, 35)

In [126]:
# keep most frequent shape for each segment_id
df = (df.sort_values(['segment_id', 'n_trips_sch'])
      .drop_duplicates(subset = ['segment_id', 'schedule_gtfs_dataset_key'], keep='last')
      )

In [127]:
df.shape

(35158, 35)

In [128]:
shs_delay = df.merge(delay_by_segment, on = ['segment_id', 'schedule_gtfs_dataset_key']).sort_values('transit_veh_hrs_delay', ascending=True)

In [129]:
to_map = shs_delay[(shs_delay.transit_veh_hrs_delay > .1) & (shs_delay.approx_shs_intersect_meters >= 200)]

In [130]:
to_map.shape

(9919, 37)

In [131]:
to_map.columns

Index(['schedule_gtfs_dataset_key', 'shape_id', 'shape_array_key', 'route_id',
       'direction_id', 'stop_pair', 'segment_id', 'stop_pair_name',
       'time_of_day', 'p50_mph', 'n_trips', 'p20_mph', 'p80_mph',
       'n_trips_sch', 'trips_hr_sch', 'route_short_name', 'geometry', 'name',
       'base64_url', 'caltrans_district', 'analysis_name', 'source_record_id',
       'index_right', 'Route', 'County', 'District', 'RouteType', 'NB', 'SB',
       'EB', 'WB', 'highway_length', 'approx_shs_intersect_meters',
       'fast_slow_ratio', 'analysis_date', 'total_transit_delay_sec',
       'transit_veh_hrs_delay'],
      dtype='object')

In [132]:
shs_delay.query('analysis_name.str.contains("San Diego") & route_short_name == "20" & segment_id == "94043-99998-7"')[map_cols]

,analysis_name,transit_veh_hrs_delay,n_trips,n_trips_sch,segment_id,route_short_name,geometry,time_of_day,p20_mph,p50_mph,p80_mph,Route,District,RouteType,name
33767,"San Diego Metropolitan Transit System, Airport...",1.16,24,29,94043-99998-7,20,"POLYGON ((267114.020 -573743.228, 267158.905 -...",All Day,39.0,54.6,60.5,805,11,Interstate,San Diego Schedule


In [133]:
map_cols = ['analysis_name', 'transit_veh_hrs_delay', 'n_trips',
             'n_trips_sch', 'segment_id',
            'route_short_name', 'geometry', 'time_of_day',
            'p20_mph', 'p50_mph', 'p80_mph',
            'Route', 'District', 'RouteType',
            'name'
]

In [ ]:
to_map[map_cols].explore(column='transit_veh_hrs_delay', scheme='Quantiles')

2